In [1]:
# version 7 2025.12.04 ~ 05

import torch
print(torch.cuda.is_available())  # True면 정상
print(torch.cuda.get_device_name(0))  # GPU 이름 출력

True
NVIDIA GeForce GTX 1660 SUPER


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA GeForce GTX 1660 SUPER


In [ ]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
import unicodedata
import emoji
from tqdm import tqdm
from gensim.models import FastText
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer7:

    def __init__(self, data_dir="../data", images_dir="../images", results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.metrics = {}
        self.setup_result = None

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ---------------------- TEXT CLEANING ----------------------
    def _clean_text(self, text):
        text = str(text).lower()
        text = unicodedata.normalize("NFKD", text)
        text = emoji.replace_emoji(text, replace="")
        text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\$?\d+(\.\d+)?", " pricenum ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _normalize_slang(self, text):
        slang = {
            "bnwt": "brand new", "nwt": "new with tag", "nwot": "new without tag",
            "nib": "new in box", "auth": "authentic", "lg": "large",
            "sz": "size", "med": "medium", "worn once": "used once"
        }
        text = str(text)
        for k, v in slang.items():
            text = re.sub(rf"\b{k}\b", v, text)
        return text

    # ---------------------- DATA LOADING ----------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.35):
        print("📂 데이터 로딩 중...")

        self.train = pd.read_csv(os.path.join(self.data_dir, train_file), sep=sep)
        self.test = pd.read_csv(os.path.join(self.data_dir, test_file), sep=sep)

        self.train = self.train[self.train["price"] > 0]
        self.train["price"] = np.log1p(self.train["price"])

        if undersample_frac:
            self._stratified_sample(frac=undersample_frac)

        for df in [self.train, self.test]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x else ["missing"] * 3)
                )
            )
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")
            df["name"] = df["name"].fillna("No name")
            df.drop(columns=["category_name"], inplace=True)

            df["name"] = df["name"].apply(lambda x: self._normalize_slang(self._clean_text(x)))
            df["item_description"] = df["item_description"].apply(lambda x: self._normalize_slang(self._clean_text(x)))

            df["name_len_char"] = df["name"].str.len()
            df["name_len_word"] = df["name"].str.split().str.len()
            df["desc_len_char"] = df["item_description"].str.len()
            df["desc_len_word"] = df["item_description"].str.split().str.len()
            df["has_brand_in_name"] = df.apply(lambda r: int(r["brand_name"].lower() in r["name"].lower()), axis=1)
            df["has_brand_in_desc"] = df.apply(lambda r: int(r["brand_name"].lower() in r["item_description"].lower()), axis=1)

            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        print(f"✅ 데이터 로드 완료: train={self.train.shape}, test={self.test.shape}")

    def _stratified_sample(self, frac=0.35, bins=10):
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        self.train = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        ).reset_index(drop=True)
        print(f"⚠ Stratified undersampling → {self.train.shape}")

    # ---------------------- VECTORIZATION ----------------------
    def vectorize_text(self, method="fasttext", **kwargs):
        methods = {"tfidf": self._tfidf, "fasttext": self._fasttext, "bert": self._bert}

        if method not in methods:
            raise ValueError("❌ method must be one of: tfidf / fasttext / bert")

        print(f"\n🔧 벡터화 방식: {method.upper()}")
        methods[method](**kwargs)

        self._add_tabular_features()
        print(f"📌 최종 feature shape → {self.train_vectorized.shape}")

    def _tfidf(self, n_components=150):
        vec = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=3, max_df=0.95)
        train = vec.fit_transform(self.train["item_description"])
        test = vec.transform(self.test["item_description"])

        svd = TruncatedSVD(n_components=min(n_components, train.shape[1]-1), random_state=23)
        self.train_vectorized = pd.DataFrame(svd.fit_transform(train))
        self.test_vectorized = pd.DataFrame(svd.transform(test))

    def _fasttext(self, fasttext_size=100):
        sentences = [str(x).split() for x in pd.concat([self.train["item_description"], self.test["item_description"]])]
        model = FastText(sentences, vector_size=fasttext_size, window=5, min_count=2, sg=1, workers=4)

        def vec(t): 
            words=[w for w in t.split() if w in model.wv]
            return np.mean([model.wv[w] for w in words], axis=0) if words else np.zeros(fasttext_size)

        self.train_vectorized = pd.DataFrame(np.vstack(self.train["item_description"].apply(vec)))
        self.test_vectorized = pd.DataFrame(np.vstack(self.test["item_description"].apply(vec)))

    def _bert(self, model_name="all-MiniLM-L6-v2"):
        self.train["combined"] = (
            self.train["main_cat"] + " " + self.train["sub_cat"] + " " + self.train["sub_sub_cat"] +
            " — " + self.train["name"] + " — " + self.train["item_description"]
        )
        self.test["combined"] = (
            self.test["main_cat"] + " " + self.test["sub_cat"] + " " + self.test["sub_sub_cat"] +
            " — " + self.test["name"] + " — " + self.test["item_description"]
        )

        model = SentenceTransformer(model_name, device="cuda")

        self.train_vectorized = pd.DataFrame(
            model.encode(self.train["combined"].tolist(), batch_size=64, show_progress_bar=True)
        )
        self.test_vectorized = pd.DataFrame(
            model.encode(self.test["combined"].tolist(), batch_size=64, show_progress_bar=True)
        )

    def _add_tabular_features(self):
        cols = ["main_cat","sub_cat","sub_sub_cat","brand_name","item_condition_id","shipping",
                "name_len_char","name_len_word","desc_len_char","desc_len_word",
                "has_brand_in_name","has_brand_in_desc"]

        for c in cols:
            self.train_vectorized[c] = self.train[c].reset_index(drop=True)
            self.test_vectorized[c] = self.test[c].reset_index(drop=True)


    # ---------------------- MODELING ----------------------
    def setup_pycaret(self, fold=3, use_gpu=True):
        print("\n⚙ PyCaret Setup 중...")

        self.setup_result = setup(
            data=self.train_vectorized.assign(price=self.train["price"].reset_index(drop=True)),
            target="price",
            session_id=23,
            fold=fold,
            use_gpu=use_gpu,
            normalize=True,
            transformation=False,
            silent=True,
            html=False
        )

        print("✅ Setup 완료")

    def find_and_blend_models(self):
        print("\n🤖 모델 학습 + 블렌딩 중...")
        models = [create_model(m) for m in ["lightgbm", "ridge", "catboost", "xgboost"]]
        self.best_model = blend_models(estimator_list=models, optimize="R2", choose_better=True)
        print("🏆 최종 모델 선택 완료")

    # ---------------------- EVALUATION ----------------------
    def save_metrics(self):
        pred = predict_model(self.best_model, data=self.train_vectorized)
        y_true = np.expm1(self.train["price"])
        y_pred = np.expm1(pred["prediction_label"])

        self.metrics = {
            "R2": float(r2_score(y_true, y_pred)),
            "RMSE": float(mean_squared_error(y_true, y_pred, squared=False)),
            "MAE": float(mean_absolute_error(y_true, y_pred))
        }

        path = os.path.join(self.results_dir, f"metrics_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
        json.dump(self.metrics, open(path, "w"), indent=4)

        print(f"📁 성능 저장됨 → {path}")
        print(self.metrics)

    # ---------------------- SUBMISSION ----------------------
    def predict_test(self):
        predictions = predict_model(self.best_model, data=self.test_vectorized)
        price = np.expm1(predictions["prediction_label"])

        sub = pd.DataFrame({"test_id": self.test["test_id"], "price": price})
        path = os.path.join(self.results_dir, f"submission_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        sub.to_csv(path, index=False)

        print(f"📤 제출 파일 저장됨 → {path}")
        return sub



# ---------------------- 실행 ----------------------
print("=" * 60)
print("🚀 Mercari Price Prediction - FINAL VERSION (v9)")
print("=" * 60)

analyzer = MercariPyCaretAnalyzer7()

analyzer.load_data(undersample_frac=0.35)
analyzer.vectorize_text(method="fasttext")   # or "tfidf", "bert"

analyzer.train_vectorized.reset_index(drop=True, inplace=True)
analyzer.train.reset_index(drop=True, inplace=True)

analyzer.setup_pycaret(fold=3, use_gpu=True)
analyzer.find_and_blend_models()
analyzer.save_metrics()
analyzer.predict_test()

print("\n🎉 완료!")


In [4]:
print("=" * 60)
print("Mercari Price Suggestion - v8 최종")
print("=" * 60)

analyzer = MercariPyCaretAnalyzer7()
analyzer.load_data(undersample_frac=0.35)

Mercari Price Suggestion - v8 최종
📂 데이터 로딩 중...
⚠ Stratified undersampling 적용: (518582, 9)
✅ 데이터 로드 완료: train (518582, 17), test (693359, 15)


In [ ]:
analyzer.vectorize_text(method="fasttext")   # or "tfidf", "bert"

In [ ]:
analyzer.train_vectorized.reset_index(drop=True, inplace=True)

In [ ]:
analyzer.train.reset_index(drop=True, inplace=True)

In [ ]:
analyzer.setup_pycaret(fold=3, use_gpu=True)
print("\n✅ 완료!")

In [ ]:
analyzer.find_and_blend_models()
print("\n✅ 완료!")

In [ ]:
analyzer.save_metrics()
analyzer.predict_test()
print("\n✅ 완료!")

In [ ]:
# 성능 비교 루프
methods = ["tfidf", "fasttext", "bert"]
results = []

for m in methods:
    print("\n" + "="*60)
    print(f"▶ {m.upper()} 방식 실행")
    print("="*60)

    analyzer = MercariPyCaretAnalyzer7()
    analyzer.load_data(undersample_frac=0.35)

    # 벡터화 (저장/불러오기 자동 처리)
    analyzer.vectorize_text(method=m)

    # PyCaret 환경 설정
    analyzer.setup_pycaret(fold=3,use_gpu=True)

    # 모델 학습 및 블렌딩
    analyzer.find_and_blend_models(use_kaggle_winners=True)

    # 성능 지표 저장
    analyzer.save_metrics(model_name=m)

    # 결과 기록
    results.append({
        "Method": m,
        "R2": analyzer.metrics["R2"],
        "RMSE": analyzer.metrics["RMSE"],
        "MAE": analyzer.metrics["MAE"]
    })

# 결과 테이블 출력
results_df = pd.DataFrame(results)
print("\n📊 성능 비교 결과")
print(results_df)